# Shreve Week 03 — Markov Property, Reflection & Passage Times

**Week 3** — Markov Property, Reflection & Passage Times

This notebook teaches **markov property, reflection & passage times** in the style of our graduate probability notebook: definitions from **Shreve**, intuition and examples from **Baxter & Rennie**, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve | Baxter & Rennie |
|------|-------|--------|-----------------|
| 1 | **Markov property** | Ch. 3.3 | Ch. 3.1 |
| 2 | **Reflection principle** | Ch. 3.3.6 | Ch. 3.1 |
| 3 | **First passage times** | Ch. 3.3.6 | Ch. 3.1 |
| 4 | **Maximum of Brownian motion** | Ch. 3.3.6 | Ch. 3.1 |
| — | **Problem set** | Ch. 3 | App. 3 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. **Shreve** (*Stochastic Calculus for Finance II*) — rigorous measure-theoretic treatment; see chapter pointers in each section.
5. **Baxter & Rennie** (*Financial Calculus*) — market intuition, replication, and worked examples; see spotlight sections.

**References:**
- **Shreve** Vol II — Ch. 3 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Baxter & Rennie**, *Financial Calculus* — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Markov Property

$W_t$ is **Markov:** $P(W_t \in B \mid \mathcal{F}_s) = P(W_t \in B \mid W_s)$ for $t > s$.

Future depends only on present, not full history. Equivalently: $W_t - W_s$ is independent of $\mathcal{F}_s$.

**Shreve Ch. 3.3:** Markov property of Brownian motion.


In [ ]:
# Markov check: W_t - W_s independent of W_s
rng = np.random.default_rng(20)
n = 200_000
s, t = 0.3, 1.0
Ws = rng.normal(0, np.sqrt(s), size=n)
increment = rng.normal(0, np.sqrt(t - s), size=n)
Wt = Ws + increment

# Correlation between W_s and (W_t - W_s) should be ~0
corr = np.corrcoef(Ws, Wt - Ws)[0, 1]
print(f"Corr(W_s, W_t - W_s) = {corr:.4f} (theory 0)")


---
# Part 2 — Reflection Principle

If path hits level $a > 0$ before time $T$, reflect the path after first hitting $a$. Reflection preserves increments distribution.

**Key result:** $P(\max_{0 \leq t \leq T} W_t \geq a) = 2 P(W_T \geq a)$ for $a > 0$.

**Shreve Ch. 3.3.6:** reflection principle.


In [ ]:
# Reflection principle: P(max W_t >= a) vs 2*P(W_T >= a)
rng = np.random.default_rng(21)
T, a = 1.0, 0.5
n_sims, n_steps = 50_000, 500
dt = T / n_steps

max_W = np.zeros(n_sims)
W_T = np.zeros(n_sims)
for i in range(n_sims):
    dW = rng.normal(0, np.sqrt(dt), size=n_steps)
    W = np.cumsum(dW)
    max_W[i] = np.max(W)
    W_T[i] = W[-1]

p_max = (max_W >= a).mean()
p_tail = 2 * (W_T >= a).mean()
print(f"P(max W >= {a})     = {p_max:.4f}")
print(f"2 * P(W_T >= {a})   = {p_tail:.4f}")
print(f"Theory 2*P(W_T>=a)  = {2*(1-stats.norm.cdf(a/np.sqrt(T))):.4f}")


---
# Part 3 — First Passage Times

**First passage time** to level $a > 0$: $\tau_a = \inf\{t \geq 0 : W_t = a\}$.

Distribution: $P(\tau_a \leq t) = 2 P(W_t \geq a) = 2\left(1 - \Phi(a/\sqrt{t})\right)$.

PDF: $f_{\tau_a}(t) = \frac{a}{\sqrt{2\pi t^3}} e^{-a^2/(2t)}$ (inverse Gaussian / Lévy).

**Shreve Ch. 3.3.6:** passage time distribution.


In [ ]:
def first_passage_sim(a=1.0, T=2.0, n_steps=1000, n_sims=20_000, seed=22):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    tau = np.full(n_sims, T)  # default: not hit by T
    for i in range(n_sims):
        dW = rng.normal(0, np.sqrt(dt), size=n_steps)
        W = np.cumsum(dW)
        hit = np.where(W >= a)[0]
        if len(hit) > 0:
            tau[i] = hit[0] * dt
    return tau

a = 1.0
tau = first_passage_sim(a=a)
print(f"P(tau_a <= 1) sim = {(tau <= 1).mean():.4f}")
print(f"Theory            = {2*(1-stats.norm.cdf(a/1.0)):.4f}")

fig, ax = plt.subplots()
ax.hist(tau[tau < 2], bins=50, density=True, alpha=0.6, label="simulated")
t_plot = np.linspace(0.01, 2, 200)
pdf = a / np.sqrt(2 * np.pi * t_plot**3) * np.exp(-a**2 / (2 * t_plot))
ax.plot(t_plot, pdf, "r-", lw=2, label="theory")
ax.set_title(f"First passage time to a={a}")
ax.legend()
plt.show()


---
# Part 4 — Maximum of Brownian Motion

$M_T = \max_{0 \leq t \leq T} W_t$ has PDF $f(x) = \frac{2}{\sqrt{2\pi T}} e^{-x^2/(2T)}$ for $x \geq 0$ (half-normal).

**Shreve Ch. 3.3.6:** distribution of maximum via reflection.


In [ ]:
rng = np.random.default_rng(23)
T = 1.0
n_sims, n_steps = 30_000, 500
dt = T / n_steps
max_W = np.array([
    np.max(np.cumsum(rng.normal(0, np.sqrt(dt), n_steps)))
    for _ in range(n_sims)
])

fig, ax = plt.subplots()
ax.hist(max_W, bins=50, density=True, alpha=0.6)
x = np.linspace(0, 3, 100)
pdf = 2 / np.sqrt(2 * np.pi * T) * np.exp(-x**2 / (2 * T))
ax.plot(x, pdf, "r-", lw=2)
ax.set_title("Distribution of max W_t on [0,T]")
plt.show()


**Your turn:** Use reflection to derive $P(\tau_a \leq t)$ in terms of $W_t$.


---
## Baxter & Rennie spotlight — Properties of Brownian motion (Ch. 3.1)

B&R develop **continuous processes** with the **Markov property**: the future of $W_t$ depends only on the present value, not the path history — matching Shreve's reflection-principle setup.

They stress **quadratic variation** $[W]_t = t$ as the distinguishing feature of Brownian motion (smooth functions have zero QV).

**Read:** B&R Ch. 3.1 sections on Brownian motion before moving to stochastic calculus in Ch. 3.2.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Prove $P(\max_{0\leq t\leq T} W_t \geq a) = 2P(W_T \geq a)$ using reflection.

2. Find $E[\tau_a]$ for first passage to $a>0$ (it is infinite!).

3. For $a<0$, relate $P(\min W_t \leq a)$ to a normal tail.

4. Show the Markov property implies $W_{t_3} - W_{t_2}$ is independent of $W_{t_1}$ for $t_1 < t_2 < t_3$.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Reflect paths that hit $a$ before $T$; bijection gives equal mass above $a$ at $T$ from paths that hit $a$ and paths that end above $a$.

**2.** $E[\tau_a] = \infty$ (BM hits every level but expected time is infinite).

**3.** By symmetry: $P(\min W_t \leq -a) = 2P(W_T \leq -a)$.

**4.** Increment $W_{t_3}-W_{t_2}$ independent of $\mathcal{F}_{t_2}$, which contains $W_{t_1}$.

</details>


---
## Further reading

### Shreve
- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 3 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

### Baxter & Rennie
- **Baxter & Rennie**, Ch. 3.1 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf): Markov property, quadratic variation, transition probabilities.
- Shreve Ch. 3.3.6 adds **reflection principle** detail; B&R builds the same intuition with less measure theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
